In [6]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

In [7]:
ARTIFACTS = Path("../artifacts")

train = pd.read_parquet(ARTIFACTS / "train.parquet")
validation = pd.read_parquet(ARTIFACTS / "validation.parquet")
test = pd.read_parquet(ARTIFACTS / "test.parquet")

print("Train:", train.shape)
print("Validation:", validation.shape)
print("Test:", test.shape)

Train: (67533, 20)
Validation: (14471, 20)
Test: (14472, 20)


In [8]:
# ########
# order_purchase_timestamp ✅
# order_approved_at ⚠️ 
# order_delivered_carrier_date ❌ 
# order_delivered_customer_date ❌ 
# order_estimated_delivery_date ⚠️ 
# order aggregates ✅
# customer state ✅
# label ❌ 
# IDs ❌
# #########

# Create time features

In [9]:
def create_features(df):
    df = df.copy()

    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_day"] = df["order_purchase_timestamp"].dt.day
    df["purchase_weekofyear"] = df["order_purchase_timestamp"].dt.isocalendar().week.astype(int)

    # Weekend indicator
    df["is_weekend"] = (
        df["purchase_dayofweek"] >= 5
    ).astype(int)

    return df

In [10]:
train_fe = create_features(train)
validation_fe = create_features(validation)
test_fe = create_features(test)

In [11]:
def create_delivery_features(df):
    df = df.copy()

    df["estimated_delivery_days"] = (
        df["order_estimated_delivery_date"]
        - df["order_purchase_timestamp"]
    ).dt.total_seconds() / (24 * 60 * 60)

    return df

In [12]:
train_fe = create_delivery_features(train_fe)
validation_fe = create_delivery_features(validation_fe)
test_fe = create_delivery_features(test_fe)

In [13]:
numeric_features = [
    "item_count",
    "product_count",
    "seller_count",
    "total_price",
    "total_freight",
    "payment_count",
    "payment_value",
    "max_installments",
    "purchase_hour",
    "purchase_dayofweek",
    "purchase_month",
    "purchase_day",
    "purchase_weekofyear",
    "is_weekend",
    "estimated_delivery_days"
]

In [14]:
categorical_features = [
    "customer_state"
]

# Create X and y

In [15]:
X_train = train_fe[numeric_features + categorical_features].copy()
y_train = train_fe["label"].copy()

X_validation = validation_fe[numeric_features + categorical_features].copy()
y_validation = validation_fe["label"].copy()

X_test = test_fe[numeric_features + categorical_features].copy()
y_test = test_fe["label"].copy()

In [16]:
print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

X_train: (67533, 16)
X_validation: (14471, 16)
X_test: (14472, 16)


# Handle missing values and encode categories

In [17]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore",
        sparse_output=False
    ))
])

# Build the transformer

In [18]:
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

In [19]:
X_train_transformed = preprocessor.fit_transform(X_train)

In [20]:
X_validation_transformed = preprocessor.transform(X_validation)

X_test_transformed = preprocessor.transform(X_test)

In [21]:
feature_names = preprocessor.get_feature_names_out()

print("Number of final features:", len(feature_names))
print("Final features:", feature_names)

Number of final features: 42
Final features: ['numeric__item_count' 'numeric__product_count' 'numeric__seller_count'
 'numeric__total_price' 'numeric__total_freight' 'numeric__payment_count'
 'numeric__payment_value' 'numeric__max_installments'
 'numeric__purchase_hour' 'numeric__purchase_dayofweek'
 'numeric__purchase_month' 'numeric__purchase_day'
 'numeric__purchase_weekofyear' 'numeric__is_weekend'
 'numeric__estimated_delivery_days' 'categorical__customer_state_AC'
 'categorical__customer_state_AL' 'categorical__customer_state_AM'
 'categorical__customer_state_AP' 'categorical__customer_state_BA'
 'categorical__customer_state_CE' 'categorical__customer_state_DF'
 'categorical__customer_state_ES' 'categorical__customer_state_GO'
 'categorical__customer_state_MA' 'categorical__customer_state_MG'
 'categorical__customer_state_MS' 'categorical__customer_state_MT'
 'categorical__customer_state_PA' 'categorical__customer_state_PB'
 'categorical__customer_state_PE' 'categorical__customer

In [22]:
X_train_final = pd.DataFrame(
    X_train_transformed,
    columns=feature_names,
    index=train_fe.index
)

X_validation_final = pd.DataFrame(
    X_validation_transformed,
    columns=feature_names,
    index=validation_fe.index
)

X_test_final = pd.DataFrame(
    X_test_transformed,
    columns=feature_names,
    index=test_fe.index
)

In [23]:
display(X_train_final.head())

,numeric__item_count,numeric__product_count,numeric__seller_count,numeric__total_price,numeric__total_freight,numeric__payment_count,numeric__payment_value,numeric__max_installments,numeric__purchase_hour,numeric__purchase_dayofweek,...,categorical__customer_state_PR,categorical__customer_state_RJ,categorical__customer_state_RN,categorical__customer_state_RO,categorical__customer_state_RR,categorical__customer_state_RS,categorical__customer_state_SC,categorical__customer_state_SE,categorical__customer_state_SP,categorical__customer_state_TO
0,3.430235,-0.169718,-0.104985,-0.004457,-0.687732,-0.11989,-0.251905,-0.353682,-0.519162,0.106861,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,-0.262316,-0.169718,-0.104985,-0.516529,-0.334222,-0.11989,-0.525992,-0.716662,-1.079511,-1.418102,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.262316,-0.169718,-0.104985,-0.555518,-0.252720,-0.11989,-0.555721,-0.716662,0.227972,-1.418102,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,-0.262316,-0.169718,-0.104985,-0.557467,-0.406724,-0.11989,-0.571961,-0.716662,1.161888,-1.418102,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,-0.262316,-0.169718,-0.104985,-0.484411,-0.250220,-0.11989,-0.487397,-0.716662,1.161888,-1.418102,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [24]:
train_final = X_train_final.copy()
train_final["label"] = y_train.values

validation_final = X_validation_final.copy()
validation_final["label"] = y_validation.values

test_final = X_test_final.copy()
test_final["label"] = y_test.values

In [25]:
(ARTIFACTS / "features").mkdir(exist_ok=True)

In [26]:
train_final.to_parquet(
    ARTIFACTS / "features" / "train_features.parquet"
)

validation_final.to_parquet(
    ARTIFACTS / "features" / "validation_features.parquet"
)

test_final.to_parquet(
    ARTIFACTS / "features" / "test_features.parquet"
)

In [27]:
joblib.dump(
    preprocessor,
    ARTIFACTS / "features" / "preprocessor.joblib"
)

['../artifacts/features/preprocessor.joblib']

In [28]:
preprocessor = joblib.load(
    ARTIFACTS / "features" / "preprocessor.joblib"
)

In [29]:
feature_list = pd.DataFrame({
    "feature": feature_names
})

feature_list.to_csv(
    ARTIFACTS / "features" / "feature_list.csv",
    index=False
)

In [30]:
print("Train:", train_final.shape)
print("Validation:", validation_final.shape)
print("Test:", test_final.shape)

print("\nMissing values:")
print("Train:", train_final.isna().sum().sum())
print("Validation:", validation_final.isna().sum().sum())
print("Test:", test_final.isna().sum().sum())

print("\nArtifacts:")
for path in (ARTIFACTS / "features").iterdir():
    print(path.name)

Train: (67533, 43)
Validation: (14471, 43)
Test: (14472, 43)

Missing values:
Train: 0
Validation: 0
Test: 0

Artifacts:
validation_features.parquet
preprocessor.joblib
train_features.parquet
feature_list.csv
test_features.parquet


In [31]:
print("Feature engineering completed successfully.")

print(f"Number of features: {len(feature_names)}")
print(f"Train shape: {train_final.shape}")
print(f"Validation shape: {validation_final.shape}")
print(f"Test shape: {test_final.shape}")

assert len(feature_names) == 42
assert train_final.shape[1] == 43
assert validation_final.shape[1] == 43
assert test_final.shape[1] == 43

assert train_final.isna().sum().sum() == 0
assert validation_final.isna().sum().sum() == 0
assert test_final.isna().sum().sum() == 0

print("All validation checks passed.")

Feature engineering completed successfully.
Number of features: 42
Train shape: (67533, 43)
Validation shape: (14471, 43)
Test shape: (14472, 43)
All validation checks passed.


## Conclusion

Feature engineering was completed using only information available at prediction
time. Future delivery information, including the actual delivery dates and carrier
delivery date, was excluded to prevent target leakage.

Time-based features were derived from the purchase timestamp, including purchase
hour, day of week, month, day, week of year, and weekend indicator. The estimated
delivery window was also calculated using the purchase timestamp and estimated
delivery date.

Missing numerical values were handled using median imputation, while the customer
state categorical feature was handled using most-frequent imputation and one-hot
encoding. Numerical features were standardized using `StandardScaler`.

All preprocessing transformations were fitted using the training data only and then
applied to the validation and test sets. This prevents information from the
validation or test sets from influencing the preprocessing process.

The final feature set contains 42 features. The resulting training, validation, and
test datasets contain 67,533, 14,471, and 14,472 rows respectively, with no remaining
missing values.

The fitted preprocessing pipeline was saved as `preprocessor.joblib`, and the final
feature names were saved in `feature_list.csv`. These artifacts allow the same
transformations to be reused in production without fitting the preprocessing steps
again on new data.